Configure HLS Walkthrough

This notebook explains how `Driver/CartPoleSimulation/SI_Toolkit_ASF/config_hls.yml` controls hls4ml conversion in the CartPole workflow.

**Config file:** `config_hls.yml`

## 0) Resolve paths and load config


In [ ]:
import os
import sys
import shutil
import yaml
from pathlib import Path
from typing import Optional
from datetime import datetime

def find_repo_root(start: Optional[Path] = None) -> Path:
    cursor = (start or Path.cwd()).resolve()
    for candidate in [cursor, *cursor.parents]:
        if (candidate / "Driver" / "CartPoleSimulation").exists() and (candidate / "README-demo-hls4ml-25.md").exists():
            return candidate
    raise FileNotFoundError("Could not locate physical-cartpole repo root")

REPO = find_repo_root()
SIM = REPO / "Driver" / "CartPoleSimulation"
SI_ASF = SIM / "SI_Toolkit_ASF"
HLS_CONFIG = SI_ASF / "config_hls.yml"

print("REPO:", REPO)
print("SIM:", SIM)
print("HLS_CONFIG:", HLS_CONFIG)

with HLS_CONFIG.open("r") as f:
    cfg = yaml.safe_load(f)

## 1) Look at the current values


In [ ]:
for k in [
    "board",
    "part",
    "path_to_hls_installation",
    "path_to_models",
    "net_name",
    "batch_size",
    "Strategy",
    "ReuseFactor",
    "backend",
    "with_brunton_testing",
    "output_dir",
]:
    print(f"{k}: {cfg.get(k)}")

print("\nPRECISION:")
for k, v in cfg.get("PRECISION", {}).items():
    print(f"  {k}: {v}")


## 2) Parameters 

### 2.1 Model and path selection
- `path_to_models`: model root folder used by `get_net(...)` during conversion.
- `net_name`: specific model folder under `path_to_models`.
- `batch_size`: inference batch size passed into `get_net(...)`.

### 2.2 Numeric precision
- `PRECISION.input_and_output`: IO precision (first/last layer results).
- `PRECISION.activations`: activation layer output precision.
- `PRECISION.weights_and_biases`: parameter precision.
- `PRECISION.intermediate_results`: default internal accumulation/result precision.

In general:
- wider precision -> better accuracy, higher area
- narrower precision -> lower area, increased error risk

### 2.3 Scheduling and resource tradeoffs
- `Strategy`: `Resources` vs `Latency` style optimization.
- `ReuseFactor`: larger value increases the hardware componets that get reused (lower area, higher latency).

### 2.4 Tool/backend/target
- `backend`: hls4ml backend (here usually `Vivado`).
- `part` and `board`: FPGA target details.
- `path_to_hls_installation`: Vivado `bin` path inserted into `PATH`.

### 2.5 Output layout
- `output_dir`: generated hls4ml project/report folder.


## 3) Apply config edits (backup & write)

Set overrides in the next cell, then run write.


In [ ]:
# Edit these parameters to override configs (set to None to keep existing config)
OVERRIDE_NET_NAME = None
OVERRIDE_MODELS_DIR = None   # absolute path or path relative to SIM
OVERRIDE_OUTPUT_NAME = None  # folder under REPO/HLS4ML
OVERRIDE_REUSE_FACTOR = None
OVERRIDE_STRATEGY = None     # "Resources" or "Latency"
OVERRIDE_PRECISION = None    # example: {"input_and_output": "ap_fixed<12,2>", ...}

# Resolve Vivado from env first 
vivado_root = os.environ.get("XILINX_VIVADO", "").strip()
if vivado_root:
    resolved_vivado_bin = str((Path(vivado_root) / "bin").resolve())
else:
    resolved_vivado_bin = cfg["path_to_hls_installation"]

resolved_net_name = OVERRIDE_NET_NAME or cfg["net_name"]

if OVERRIDE_MODELS_DIR is None:
    resolved_models_rel = cfg["path_to_models"]
else:
    candidate = Path(OVERRIDE_MODELS_DIR)
    if not candidate.is_absolute():
        candidate = (SIM / candidate).resolve()
    resolved_models_rel = os.path.relpath(candidate, SIM)

if OVERRIDE_OUTPUT_NAME is None:
    resolved_output_rel = cfg["output_dir"]
else:
    resolved_output_rel = os.path.relpath((REPO / "HLS4ML" / OVERRIDE_OUTPUT_NAME).resolve(), SIM)

print("resolved_vivado_bin:", resolved_vivado_bin)
print("resolved_models_rel:", resolved_models_rel)
print("resolved_net_name:", resolved_net_name)
print("resolved_output_rel:", resolved_output_rel)


In [ ]:
RUN_TAG = datetime.now().strftime("%Y%m%d_%H%M%S")
backup_path = HLS_CONFIG.with_suffix(f".yml.bak_{RUN_TAG}")
shutil.copy2(HLS_CONFIG, backup_path)

cfg_out = dict(cfg)
cfg_out["path_to_hls_installation"] = resolved_vivado_bin
cfg_out["path_to_models"] = resolved_models_rel
cfg_out["net_name"] = resolved_net_name
cfg_out["output_dir"] = resolved_output_rel

if OVERRIDE_REUSE_FACTOR is not None:
    cfg_out["ReuseFactor"] = int(OVERRIDE_REUSE_FACTOR)
if OVERRIDE_STRATEGY is not None:
    cfg_out["Strategy"] = OVERRIDE_STRATEGY
if OVERRIDE_PRECISION is not None:
    cfg_out["PRECISION"] = OVERRIDE_PRECISION

with HLS_CONFIG.open("w") as f:
    yaml.safe_dump(cfg_out, f, sort_keys=False)

print("Backup:", backup_path)
print("Wrote:", HLS_CONFIG)
